# Wigner $W_l$ Parameters

The third-order Steinhardt invariant $W_l$ ([Steinhardt, Nelson & Ronchetti, Phys. Rev. B **28**, 784, 1983](https://doi.org/10.1103/PhysRevB.28.784)) provides additional structural discrimination beyond the second-order $q_l$. It is constructed from the same spherical-harmonic coefficients but contracted with **Wigner 3j symbols**:

$$W_l(i) = \sum_{\substack{m_1, m_2, m_3 \\ m_1+m_2+m_3=0}} \begin{pmatrix} l & l & l \\ m_1 & m_2 & m_3 \end{pmatrix} q_{lm_1}(i)\, q_{lm_2}(i)\, q_{lm_3}(i)$$

The **normalised** form $\hat{W}_l$ divides by $(\sum_m |q_{lm}|^2)^{3/2}$, making it independent of the magnitude of bond orientational order.

A key property: **$\hat{W}_6$ has opposite sign for FCC (−0.013) and BCC (+0.013)**, making it ideal for distinguishing these two structures where $q_6$ alone cannot.

Neighbor-averaged versions $\bar{W}_l$ ([Lechner & Dellago, J. Chem. Phys. **129**, 114707, 2008](https://doi.org/10.1063/1.2977970)) replace $q_{lm}$ with $\bar{q}_{lm}$ — the average over an atom and its neighbors — further sharpening structural discrimination.

In [ ]:
import pyscal
import numpy as np
from pyscal.structures import make_crystal

## Basic Calculation

Compute $\hat{W}_4$ and $\hat{W}_6$ for a perfect FCC crystal.

In [ ]:
fcc = make_crystal("fcc", lattice_constant=4.05, repetitions=(4, 4, 4))
pyscal.find_neighbors(fcc, method="cutoff", cutoff=0)

w4, w6 = pyscal.wigner_w_parameter(fcc, l=[4, 6])

print(f"FCC: W-hat_4 = {w4.mean():.5f}, W-hat_6 = {w6.mean():.5f}")

## Comparing Crystal Structures

$\hat{W}_6$ distinguishes FCC from BCC (opposite sign), whereas $q_6$ gives similar values for both. This is the key advantage of the third-order invariant.

In [ ]:
structures = {
    "fcc": make_crystal("fcc", lattice_constant=4.05, repetitions=(4, 4, 4)),
    "bcc": make_crystal("bcc", lattice_constant=2.87, repetitions=(4, 4, 4)),
    "hcp": make_crystal("hcp", lattice_constant=3.21, repetitions=(4, 4, 4)),
}

print(f"{'Structure':<10} {'W-hat_4':>10} {'W-hat_6':>10} {'q_6':>8}")
print("-" * 40)

for name, atoms in structures.items():
    pyscal.find_neighbors(atoms, method="cutoff", cutoff=0)
    w4, w6 = pyscal.wigner_w_parameter(atoms, l=[4, 6])
    [q6] = pyscal.steinhardt_parameter(atoms, l=6)
    print(f"{name:<10} {w4.mean():>10.5f} {w6.mean():>10.5f} {q6.mean():>8.4f}")

### Reference values (Steinhardt 1983, Table I)

| Structure | $\hat{W}_4$ | $\hat{W}_6$ | $q_6$ |
|---|---|---|---|
| FCC | −0.15932 | −0.01316 | 0.575 |
| BCC | +0.15932 | +0.01316 | 0.511 |
| HCP | +0.13410 | −0.01244 | 0.485 |
| Icosahedral | — | −0.16975 | 0.663 |

## Odd $l$ Values

For $j_1=j_2=j_3=l$, the Wigner 3j symbol requires $3l$ to be even. So $W_l$ vanishes identically for odd $l$.

In [ ]:
fcc = make_crystal("fcc", lattice_constant=4.05, repetitions=(3, 3, 3))
pyscal.find_neighbors(fcc, method="cutoff", cutoff=0)

for l in [3, 4, 5, 6, 7, 8]:
    [w] = pyscal.wigner_w_parameter(fcc, l=l)
    print(f"l={l}: W-hat_{l} = {w.mean():+.5f}  {'(zero - odd l)' if l % 2 != 0 else ''}")

## Averaged $\bar{W}_l$ (Lechner–Dellago)

The neighbor-averaged variant replaces $q_{lm}(i)$ with $\bar{q}_{lm}(i)$, giving smoother distributions and better structural discrimination in systems at finite temperature.

In [ ]:
# Compare regular and averaged W-hat on a noisy FCC structure
noisy_fcc = make_crystal("fcc", lattice_constant=4.05, repetitions=(4, 4, 4), noise=0.1)
pyscal.find_neighbors(noisy_fcc, method="cutoff", cutoff=0)

# Regular
[w6] = pyscal.wigner_w_parameter(noisy_fcc, l=6)

# Averaged
[w6_avg] = pyscal.wigner_w_parameter(noisy_fcc, l=6, averaged=True)

print(f"{'':>18} {'mean':>10} {'std':>10}")
print(f"{'W-hat_6':<18} {w6.mean():>10.5f} {w6.std():>10.5f}")
print(f"{'avg W-hat_6':<18} {w6_avg.mean():>10.5f} {w6_avg.std():>10.5f}")
print(f"\nAveraging reduces std by {w6.std()/w6_avg.std():.1f}x")

## Unnormalised $W_l$

By default, `wigner_w_parameter` returns the normalised $\hat{W}_l$. Setting `normalized=False` gives the raw $W_l$.

In [ ]:
fcc = make_crystal("fcc", lattice_constant=4.05, repetitions=(3, 3, 3))
pyscal.find_neighbors(fcc, method="cutoff", cutoff=0)

[w6_hat] = pyscal.wigner_w_parameter(fcc, l=6, normalized=True)
[w6_raw] = pyscal.wigner_w_parameter(fcc, l=6, normalized=False)

print(f"W-hat_6 (normalised): {w6_hat.mean():.5f}")
print(f"W_6    (raw):         {w6_raw.mean():.8f}")

## FCC vs BCC Discrimination

The $(\hat{W}_4, \hat{W}_6)$ plane provides a clear two-dimensional fingerprint that separates FCC from BCC without ambiguity.

In [ ]:
# Mix FCC and BCC atoms
fcc = make_crystal("fcc", lattice_constant=4.05, repetitions=(3, 3, 3), noise=0.02)
bcc = make_crystal("bcc", lattice_constant=2.87, repetitions=(3, 3, 3), noise=0.02)

for atoms_i in [fcc, bcc]:
    pyscal.find_neighbors(atoms_i, method="cutoff", cutoff=0)
    pyscal.wigner_w_parameter(atoms_i, l=[4, 6])

print(f"{'Label':<6} {'W-hat_4':>10} {'W-hat_6':>10}")
print("-" * 28)
for label, atoms_i in [("FCC", fcc), ("BCC", bcc)]:
    w4 = atoms_i.arrays["pyscal_what4"]
    w6 = atoms_i.arrays["pyscal_what6"]
    for i in range(3):
        print(f"{label:<6} {w4[i]:>10.5f} {w6[i]:>10.5f}")
    print("  ...")

## Stored Results

After calling `wigner_w_parameter`, results are stored in `atoms.arrays` with the `pyscal_` prefix.

In [ ]:
fcc = make_crystal("fcc", lattice_constant=4.05, repetitions=(3, 3, 3))
pyscal.find_neighbors(fcc, method="cutoff", cutoff=0)
pyscal.wigner_w_parameter(fcc, l=[4, 6], averaged=True)

print("W_l-related arrays stored in atoms.arrays:")
for key in sorted(fcc.arrays):
    if 'w' in key.lower() and 'pyscal' in key:
        print(f"  {key}: shape {fcc.arrays[key].shape}, mean = {fcc.arrays[key].mean():.5f}")

## References

1. P. J. Steinhardt, D. R. Nelson and M. Ronchetti, *Bond-orientational order in liquids and glasses*, Phys. Rev. B **28**, 784 (1983). [doi:10.1103/PhysRevB.28.784](https://doi.org/10.1103/PhysRevB.28.784)

2. W. Lechner and C. Dellago, *Accurate determination of crystal structures based on averaged local bond order parameters*, J. Chem. Phys. **129**, 114707 (2008). [doi:10.1063/1.2977970](https://doi.org/10.1063/1.2977970)